In [1]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

from bisect import bisect
from tqdm import tqdm
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Local imports
sys.path.insert(0, os.path.abspath('.'))
from rejectlearn.rejection import RejectionClassifier
from rejectlearn.poibin import PoiBin

In [2]:
dataset_name = "diabetes" 
algorithm = "xgb"  # "tree", "rf", "xgb"
mechanism = "regva"  # "naive", "cv", "regva

In [ ]:
# Define predictors and target
dataset = pd.read_csv(f"data/{dataset_name}.csv", sep=",")
X = dataset.drop(columns=["Class"]).values
y = pd.Series(dataset["Class"].values, dtype="float64")

if mechanism in ["regva", "avgva"]:
    results = {
                "rejection_rate": [],
                "coverage_rate": [],  
                "abs_error": [],  
                "signed_error": [],  
                "accuracy": [],
                "estimate": [],
                "lower_estimate":[],
                "upper_estimate":[],
        }
else:
    results = {
                "rejection_rate": [],
                "coverage_rate": [],  
                "abs_error": [],  
                "signed_error": [],  
                "accuracy": [],
                "estimate": [],
        }


for rejection_rate in tqdm([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]):

    results["rejection_rate"].append(rejection_rate)

    iter_accs = []
    iter_estimates = []
    iter_covs = []
    
    if mechanism in ["regva", "avgva"]:
        iter_lower_estimate = []
        iter_upper_estimate = []
    

    for _ in tqdm(range(500)):

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y)


        # Split the training set into a proper training set and a calibration set
        if mechanism != "naive":
            X_prop, X_cal, y_prop, y_cal = train_test_split(X_train, y_train, test_size=0.33, stratify=y_train)

        # Train the underlying classifier
        if algorithm == "tree":
            if mechanism != "naive":
                clf = DecisionTreeClassifier(min_samples_leaf=round(np.sqrt(len(y_prop))))
            else:
                clf = DecisionTreeClassifier(min_samples_leaf=round(np.sqrt(len(y_train))))
        if algorithm == "rf":
            clf = RandomForestClassifier(n_jobs=-1)
        if algorithm == "xgb":
            clf = XGBClassifier(n_jobs=-1)
        
        if mechanism == "naive":
            rc = RejectionClassifier(estimator=clf, mechanism=mechanism, cv="prefit")
            rc.fit(X_train, y_train.values)
        else:
            clf.fit(X_prop, y_prop)
            rc = RejectionClassifier(estimator=clf, mechanism=mechanism, cv="prefit")
            rc.fit(X_cal, y_cal.values)
             
        # Make predictions
        y_preds = rc.predict(X_test)
            
        # Compute probabilities
        if mechanism in ["regva", "avgva"]:
            y_probs, lower_intervals, upper_intervals = rc.predict_proba(X_test)
        else:
            y_probs = rc.predict_proba(X_test)
        
        max_probs = y_probs.max(axis=1)
        max_probs = pd.Series(max_probs).reset_index(drop=True)
        y_preds = pd.Series(y_preds).reset_index(drop=True)
        y_test = pd.Series(y_test).reset_index(drop=True)
        
        prob_threshold = max_probs.quantile(rejection_rate)
        nr_idx = pd.Series(max_probs >= prob_threshold, index=max_probs.index)
        
        # Realized accuracy
        acc = accuracy_score(y_test[nr_idx], y_preds[nr_idx])
        
        inputs = list(max_probs[nr_idx])
        pb = PoiBin(inputs)
        x_values = list(range(len(inputs) + 1))
        cdf = pb.cdf(x_values)
        estimate = bisect(cdf, 0.1) / len(inputs)

        if mechanism in ["regva", "avgva"]:
            inputs_lower = list(lower_intervals[nr_idx])
            pb = PoiBin(inputs_lower)
            x_values = list(range(len(inputs) + 1))
            cdf = pb.cdf(x_values)
            cdf_lower_bound_lower_interval = bisect(cdf, 0.1) / len(inputs_lower)
            iter_lower_estimate.append(cdf_lower_bound_lower_interval)
                        
            inputs_upper = list(upper_intervals[nr_idx])
            pb = PoiBin(inputs_upper)
            x_values = list(range(len(inputs) + 1))
            cdf = pb.cdf(x_values)
            cdf_lower_bound_upper_interval = bisect(cdf, 0.1) / len(inputs_upper)
            iter_upper_estimate.append(cdf_lower_bound_upper_interval)
        
        iter_accs.append(acc)
        iter_estimates.append(estimate)
        iter_covs.append(acc >= estimate)
    
    results["coverage_rate"].append(np.mean(iter_covs))
    results["estimate"].append(np.mean(iter_estimates))
    results["accuracy"].append(np.mean(iter_accs))
    results["abs_error"].append(np.abs(np.mean(iter_estimates) - np.quantile(iter_accs, 0.1)))
    results["signed_error"].append(np.mean(iter_estimates) - np.quantile(iter_accs, 0.1))

    if mechanism in ["regva", "avgva"]:
        results["lower_estimate"].append(np.mean(iter_lower_estimate))
        results["upper_estimate"].append(np.mean(iter_upper_estimate))

    pd.DataFrame(results).to_csv(f"lower_bound_results/{dataset_name}_{mechanism}_{algorithm}.csv", index=False)


100%|██████████| 9/9 [00:03<00:00,  2.66it/s]


{'rejection_rate': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'coverage_rate': [np.float64(1.0),
  np.float64(1.0),
  np.float64(1.0),
  np.float64(1.0),
  np.float64(1.0),
  np.float64(1.0),
  np.float64(0.0),
  np.float64(1.0),
  np.float64(1.0)],
 'abs_error': [np.float64(0.08092485549132944),
  np.float64(0.0),
  np.float64(0.08208955223880599),
  np.float64(0.058064516129032184),
  np.float64(0.04040404040404033),
  np.float64(0.025974025974025983),
  np.float64(0.03076923076923077),
  np.float64(0.13953488372093026),
  np.float64(0.10344827586206895)],
 'signed_error': [np.float64(-0.08092485549132944),
  np.float64(0.0),
  np.float64(-0.08208955223880599),
  np.float64(-0.058064516129032184),
  np.float64(-0.04040404040404033),
  np.float64(-0.025974025974025983),
  np.float64(0.03076923076923077),
  np.float64(-0.13953488372093026),
  np.float64(-0.10344827586206895)],
 'accuracy': [np.float64(0.7803468208092486),
  np.float64(0.7241379310344828),
  np.float64(0.8059701492